In [3]:
import pandas as pd
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


#Angular acceleration
from sklearn.metrics import f1_score
import math
torque = np.array(df['torque_values'].tolist())[..., np.newaxis]
angle = np.array(df['angle_values'].tolist())
time  = np.array(df['time_values'].tolist())

angle_rad = np.radians(angle)

#first derivative: angular velovity
dt = np.diff(time, axis=1) + 1e-8
dtheta = np.diff(angle, axis=1)
omega = dtheta /dt  
#second derivative: angular acceleration
domega = np.diff(omega, axis=1)
dt2 = dt[:, 1:] + 1e-8

alpha = domega/ dt2  
torque_trim =torque[:, 2:] 


x_data = np.concatenate([torque_trim, alpha[..., np.newaxis]],axis=-1)
y_data = np.array(df['class_values'].tolist())
print("x_data shape:", x_data.shape)
x_data = np.transpose(x_data, (0, 2, 1))
print("transposed x_data shape:", x_data.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)


In [7]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
import numpy as np
torque = np.array(df['torque_values'].tolist())[..., np.newaxis]
angle = np.array(df['angle_values'].tolist())
time  = np.array(df['time_values'].tolist())

angle_rad = np.radians(angle)

#first derivative: angular velovity
dt = np.diff(time, axis=1) + 1e-8
dtheta = np.diff(angle, axis=1)
omega = dtheta /dt  
#second derivative: angular acceleration
domega = np.diff(omega, axis=1)
dt2 = dt[:, 1:] + 1e-8

alpha = domega/ dt2  
torque_trim =torque[:, 2:] 


x_data = np.concatenate([torque_trim, alpha[..., np.newaxis]],axis=-1)
n_samples = x_data.shape[0]
x_data = x_data.reshape(n_samples, -1)
y_data = np.array(df['class_values'].tolist())
print("x_data shape:", x_data.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)

cv_scores_val = []
best_val_f1 = -1
best_model = None

for fold, (train_index, val_index) in enumerate(skf.split(X_train_full, y_train_full)):
    x_train_fold, x_val_fold = X_train_full[train_index], X_train_full[val_index]
    y_train_fold, y_val_fold = y_train_full[train_index], y_train_full[val_index]
    print(f"Fold{fold+1}")

    mlp = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",solver="adam",max_iter=500,random_state=42)
    mlp.fit(x_train_fold, y_train_fold)
    y_val_pred = mlp.predict(x_val_fold)
    val_f1 = f1_score(y_val_fold, y_val_pred, average="macro")
    cv_scores_val.append(val_f1)
    print(f"Validation Macro F1: {val_f1:.4f} im Fold {fold+1}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model = mlp
    
    
print(f"CV-Ergebnis - Validation Macro F1: {np.mean(cv_scores_val):.4f} ± {np.std(cv_scores_val):.4f}")
test_pred = best_model.predict(X_test)
test_f1 = f1_score(y_test, test_pred, average="macro")
print(f"test macro f1: {test_f1:.4f}")
    
        



x_data shape: (12500, 1596)
Fold1
Validation Macro F1: 0.0748 im Fold 1
Fold2
Validation Macro F1: 0.0668 im Fold 2
Fold3
Validation Macro F1: 0.0630 im Fold 3
CV-Ergebnis - Validation Macro F1: 0.0682 ± 0.0049
test macro f1: 0.0717
